In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS
import fiona

# 19 - Process Final

This notebook's objective is to process the dataset to obtain its `final`version.

## Constants

In [ ]:
ROOT = '../../'

## Load the data

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.15.csv', index_col=0)

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
list(df_regbl.columns)

## Keep only needed columns (original)

In [ ]:
cols = [
 'Rdn - Canton',
 'Rdn - ID maison',
 'EGID',
 'Rdn - Commune',
 'Rdn - Catégorie de bâtiment',
 'Rdn - Année de construction',
 'Rdn - Nombre étages',
 'Rdn - Fondation',
 'Rdn - Structure de fondation',
 'Rdn - Nombre étage sous-terrain',
 'Rdn - Terrain en pente',
 'Rdn - Ventilation contrôlée',
 'Rdn - LV95_ORDINATE',
 'Rdn - LV95_ABSZISSE',
 'Rdn - LV03_ORDINATE',
 'Rdn - LV03_ABSZISSE',
 'Rdn - Protocole de mesure',
 'Rdn - Type de mesure',
 'Rdn - Type instrument de mesure',
 'Rdn - ID local',
 'Rdn - Type de local',
 'Rdn - Séjour de personnes',
 'Rdn - Heures de séjour par an',
 'Rdn - Etage',
 'Rdn - Exposition de radon (kBqh/m3)',
 'Rdn - Incertitude (kBqh/m3)',
 'Rdn - Concentration de radon (Bq/m3)_BQ_M3',
 'Rdn - duration_days',
 'Rdn - Début_year',
 'Rdn - Début_month',
 'Rdn - Début_day',
 'Rdn - Fin_year',
 'Rdn - Fin_month',
 'Rdn - Fin_day',
  # 'RegBL - Canton',
  # 'RegBL - Nom de la commune',
 #'RegBL - Numéro d’immeuble',
 #'RegBL - Numéro officiel du bâtiment',
 'RegBL - Désignation du bâtiment',
 'Coordonnée E',
 'Coordonnée N',
 #'RegBL - Provenance des coordonnées',
 #'RegBL - Statut du bâtiment',
 'RegBL - Catégorie de bâtiment',
 'RegBL - Classe de bâtiment',
 'RegBL - Année de construction du bâtiment',
 #'RegBL - Mois de construction du bâtiment',
 'RegBL - Période de construction',
 #'RegBL - Année de démolition du bâtiment',
 'RegBL - Surface du bâtiment',
 'RegBL - Volume du bâtiment',
 'RegBL - Nombre de niveaux',
 'RegBL - Nombre de logements',
 'RegBL - Nombre de pièces d’hab. indép.',
 'RegBL - Abri de protection civile',
 #'RegBL - Surface de référence énergétique',
 'RegBL - Générateur de chaleur pour le chauffage 1',
 'RegBL - Source d’énergie / de chaleur pour le chauffage 1',
 'RegBL - Générateur de chaleur pour le chauffage 2',
 'RegBL - Source d’énergie / de chaleur pour le chauffage 2',
 "RegBL - Générateur de chaleur pour l'eau chaude 1",
 "RegBL - Source d’énergie / de chaleur pour l'eau chaude 1",
 "RegBL - Générateur de chaleur pour l'eau chaude 2",
 "RegBL - Source d’énergie / de chaleur pour l'eau chaude 2",
 'Ura - Concentration uranium',
 'Clim - Durée ensoleillement',
 'Clim - Précipitation',
 'Clim - Température',
 'UncSoil - Classe de sol',
 'Relief - Altitude (m)',
 'Miner - Certification de minergie',
 'Miner - Standard de minergie',
 'Rdn Map - Probabilité dépassement valeur réf radon',
 'Rdn Map - Index de confiance dépassement valeur réf',
#  'ArealStat - RELI',
#  'ArealStat - E_COORD',
#  'ArealStat - N_COORD',
#  'ArealStat - GMDE_ID',
#  'ArealStat - GMDE_HISTID',
#  'ArealStat - FJ85',
#  'ArealStat - FJ97',
#  'ArealStat - FJ09',
#  'ArealStat - FJ18',
#  'ArealStat - FJ25',
#  'ArealStat - METHOD25',
#  'ArealStat - REVISION25',
#  'ArealStat - AS85_72',
#  'ArealStat - AS97_72',
#  'ArealStat - AS09_72',
#  'ArealStat - AS18_72',
 'ArealStat - AS25_72',
#  'ArealStat - AS85_27',
#  'ArealStat - AS97_27',
#  'ArealStat - AS09_27',
#  'ArealStat - AS18_27',
 'ArealStat - AS25_27',
#  'ArealStat - AS85_17',
#  'ArealStat - AS97_17',
#  'ArealStat - AS09_17',
#  'ArealStat - AS18_17',
 'ArealStat - AS25_17',
#  'ArealStat - AS85_4',
#  'ArealStat - AS97_4',
#  'ArealStat - AS09_4',
#  'ArealStat - AS18_4',
 'ArealStat - AS25_4',
#  'ArealStat - LC85_27',
#  'ArealStat - LC97_27',
#  'ArealStat - LC09_27',
#  'ArealStat - LC18_27',
#  'ArealStat - LC25_27',
#  'ArealStat - LC85_6',
#  'ArealStat - LC97_6',
#  'ArealStat - LC09_6',
#  'ArealStat - LC18_6',
#  'ArealStat - LC25_6',
#  'ArealStat - LU85_46',
#  'ArealStat - LU97_46',
#  'ArealStat - LU09_46',
#  'ArealStat - LU18_46',
#  'ArealStat - LU25_46',
#  'ArealStat - LU85_10',
#  'ArealStat - LU97_10',
#  'ArealStat - LU09_10',
#  'ArealStat - LU18_10',
#  'ArealStat - LU25_10',
#  'ArealStat - LU85_4',
#  'ArealStat - LU97_4',
#  'ArealStat - LU09_4',
#  'ArealStat - LU18_4',
#  'ArealStat - LU25_4',
 'distance_to_stats',
 #'HydroWM -  GWKID',
 'HydroWM -  GWKName',
 #'HydroWM -  Flussgebiet',
 'HydroWM -  Grundwasserleitertyp',
 'HydroWM -  Naturraum',
 'HydroWM -  Grundwasserregime',
 'HydroWM -  Area',
 #'HydroWM -  SHAPE_Length',
 #'HydroWM -  SHAPE_Area',
 #'HydroWM -  Farbe',
 'Seism - ZONE',
 'Geol - LITHO',
 'Geol - PERIODE',
 'Geol - EPOQUE',
 'Geol - ETAGE',
 'Geol - LEG_TEC_1',
 'Geol - LEG_TEC_2',
 'Geol - LEG_TEC_3',
 'Geol - AQUIFERE',
 'Geol - HYDROGEOL',
 'Geol - ID_LITH',
 'Geol - LITH_PET',
 'Geol - ID_GENESE',
 'Geol - GENESE',
 'Geol - ID_TYP_ROC',
 'Geol - TYPE_ROCHE',
 'Geol - COUVERTURE',
 'Geol - Distance to fault (m)',
 'geometry',
 #'Karst - BA_ID',
 'Karst - BA_Name',
 'Karst - BA_Alt_min',
 'Karst - BA_Alt_moy',
 'Karst - BA_Alt_max',
 'Karst - BA_UStruct',
 #'Karst - BA_DD',
 #'Karst - BA_Canton',
 #'Karst - BA_Rem',
 'Karst - BA_Aqui',
 #'Karst - SHAPE_Length',
 #'Karst - SHAPE_Area',
#  'Karst - BA_Surf',
#  'Karst - UB_ID',
 'Karst - UB_Name',
 'Karst - UB_Type',
 'Karst - UB_Flux',
 'Karst - UB_Alt_min',
 'Karst - UB_Alt_moy',
 'Karst - UB_Alt_max',
#  'Shape_Length_left',
#  'Shape_Area_left',
 'Karst - FID_NK',
#  'Karst - NK_ID',
 'Karst - NK_Hdyn',
 'Karst - NK_Type',
 'Karst - NK_Level',
 'Karst - MNK_ID',
#  'Shape_Length_right',
#  'Shape_Area_right',
#  'Forest - Region_de',
 'Forest - Region_fr',
#  'Forest - Region_it',
 'Forest - Region_en',
#  'Forest - Code',
#  'Forest - Subcode',
#  'Forest - Code_Bu',
#  'Forest - Code_Fi',
#  'Forest - Shape_Leng',
#  'Forest - Shape_Area',
#  'index_right',
 'TBB - CH_CODE_HN',
 #'TBB - CH_BEZ_D',
 'TBB - CH_BEZ_F',
 #'TBB - BFS_NO',
 'TBB - NAME',
 #'TBB - KT_NO',
 #'TBB - KT_KZ',
 'TBB - FLAECHE'
]

## Keep only needed columns (Fix FM 06/03)

In [ ]:
cols = [
 'Rdn - Canton',
 'Rdn - ID maison',
 'EGID',
 'Rdn - Commune',
 'Rdn - Catégorie de bâtiment',
 'Rdn - Année de construction',
 'Rdn - Nombre étages',
 'Rdn - Fondation',
 'Rdn - Structure de fondation',
 'Rdn - Nombre étage sous-terrain',
 'Rdn - Terrain en pente',
 'Rdn - Ventilation contrôlée',
 'Rdn - LV95_ORDINATE',
 'Rdn - LV95_ABSZISSE',
 'Rdn - LV03_ORDINATE',
 'Rdn - LV03_ABSZISSE',
 'Rdn - Protocole de mesure',
 'Rdn - Type de mesure',
 'Rdn - Type instrument de mesure',
 'Rdn - ID local',
 'Rdn - Type de local',
 'Rdn - Séjour de personnes',
 'Rdn - Heures de séjour par an',
 'Rdn - Etage',
 'Rdn - Exposition de radon (kBqh/m3)',
 'Rdn - Incertitude (kBqh/m3)',
 'Rdn - Concentration de radon (Bq/m3)_BQ_M3',
 'Rdn - duration_days',
 'Rdn - Début_year',
 'Rdn - Début_month',
 'Rdn - Début_day',
 'Rdn - Fin_year',
 'Rdn - Fin_month',
 'Rdn - Fin_day',
 # 'RegBL - Canton',
 # 'RegBL - Nom de la commune',
 # 'RegBL - Numéro d’immeuble',
 # 'RegBL - Numéro officiel du bâtiment',
 'RegBL - Désignation du bâtiment',
 'Coordonnée E',
 'Coordonnée N',
 #'RegBL - Provenance des coordonnées',
 #'RegBL - Statut du bâtiment',
 'RegBL - Catégorie de bâtiment',
 'RegBL - Classe de bâtiment',
 'RegBL - Année de construction du bâtiment',
 #'RegBL - Mois de construction du bâtiment',
 'RegBL - Période de construction',
 #'RegBL - Année de démolition du bâtiment',
 'RegBL - Surface du bâtiment',
 'RegBL - Volume du bâtiment',
 'RegBL - Nombre de niveaux',
 'RegBL - Nombre de logements',
 'RegBL - Nombre de pièces d’hab. indép.',
 'RegBL - Abri de protection civile',
 'RegBL - Surface de référence énergétique',
 'RegBL - Générateur de chaleur pour le chauffage 1',
 'RegBL - Source d’énergie / de chaleur pour le chauffage 1',
 'RegBL - Générateur de chaleur pour le chauffage 2',
 'RegBL - Source d’énergie / de chaleur pour le chauffage 2',
 "RegBL - Générateur de chaleur pour l'eau chaude 1",
 "RegBL - Source d’énergie / de chaleur pour l'eau chaude 1",
 "RegBL - Générateur de chaleur pour l'eau chaude 2",
 "RegBL - Source d’énergie / de chaleur pour l'eau chaude 2",
 'Ura - Concentration uranium',
 'Clim - Durée ensoleillement',
 'Clim - Précipitation',
 'Clim - Température',
 'UncSoil - Classe de sol',
 'Relief - Altitude (m)',
 'Miner - Certification de minergie',
 'Miner - Standard de minergie',
 'Rdn Map - Probabilité dépassement valeur réf radon',
 'Rdn Map - Index de confiance dépassement valeur réf',
 #  'ArealStat - RELI',
 #  'ArealStat - E_COORD',
 #  'ArealStat - N_COORD',
 #  'ArealStat - GMDE_ID',
 #  'ArealStat - GMDE_HISTID',
 #  'ArealStat - FJ85',
 #  'ArealStat - FJ97',
 #  'ArealStat - FJ09',
 #  'ArealStat - FJ18',
 #  'ArealStat - FJ25',
 #  'ArealStat - METHOD25',
 #  'ArealStat - REVISION25',
 #  'ArealStat - AS85_72',
 #  'ArealStat - AS97_72',
 #  'ArealStat - AS09_72',
 #  'ArealStat - AS18_72',
 'ArealStat - AS25_72',
 #  'ArealStat - AS85_27',
 #  'ArealStat - AS97_27',
 #  'ArealStat - AS09_27',
 #  'ArealStat - AS18_27',
 'ArealStat - AS25_27',
 #  'ArealStat - AS85_17',
 #  'ArealStat - AS97_17',
 #  'ArealStat - AS09_17',
 #  'ArealStat - AS18_17',
 'ArealStat - AS25_17',
 #  'ArealStat - AS85_4',
 #  'ArealStat - AS97_4',
 #  'ArealStat - AS09_4',
 #  'ArealStat - AS18_4',
 'ArealStat - AS25_4',
 #  'ArealStat - LC85_27',
 #  'ArealStat - LC97_27',
 #  'ArealStat - LC09_27',
 #  'ArealStat - LC18_27',
 #  'ArealStat - LC25_27',
 #  'ArealStat - LC85_6',
 #  'ArealStat - LC97_6',
 #  'ArealStat - LC09_6',
 #  'ArealStat - LC18_6',
 #  'ArealStat - LC25_6',
 #  'ArealStat - LU85_46',
 #  'ArealStat - LU97_46',
 #  'ArealStat - LU09_46',
 #  'ArealStat - LU18_46',
 #  'ArealStat - LU25_46',
 #  'ArealStat - LU85_10',
 #  'ArealStat - LU97_10',
 #  'ArealStat - LU09_10',
 #  'ArealStat - LU18_10',
 #  'ArealStat - LU25_10',
 #  'ArealStat - LU85_4',
 #  'ArealStat - LU97_4',
 #  'ArealStat - LU09_4',
 #  'ArealStat - LU18_4',
 #  'ArealStat - LU25_4',
 'distance_to_stats',
 #'HydroWM -  GWKID',
 'HydroWM -  GWKName',
 'HydroWM -  Flussgebiet',
 'HydroWM -  Grundwasserleitertyp',
 'HydroWM -  Naturraum',
 'HydroWM -  Grundwasserregime',
 'HydroWM -  Area',
 'HydroWM -  SHAPE_Length',
 #'HydroWM -  SHAPE_Area',
 'HydroWM -  Farbe',
 'Seism - ZONE',
 'Geol - LITHO',
 'Geol - PERIODE',
 'Geol - EPOQUE',
 'Geol - ETAGE',
 'Geol - LEG_TEC_1',
 'Geol - LEG_TEC_2',
 'Geol - LEG_TEC_3',
 'Geol - AQUIFERE',
 'Geol - HYDROGEOL',
 'Geol - ID_LITH',
 'Geol - LITH_PET',
 'Geol - ID_GENESE',
 'Geol - GENESE',
 'Geol - ID_TYP_ROC',
 'Geol - TYPE_ROCHE',
 'Geol - COUVERTURE',
 'Geol - Distance to fault (m)',
 'geometry',
 #'Karst - BA_ID',
 'Karst - BA_Name',
 'Karst - BA_Alt_min',
 'Karst - BA_Alt_moy',
 'Karst - BA_Alt_max',
 'Karst - BA_UStruct',
 'Karst - BA_DD',
 #'Karst - BA_Canton',
 'Karst - BA_Rem',
 'Karst - BA_Aqui',
 'Karst - SHAPE_Length',
 'Karst - SHAPE_Area',
 'Karst - BA_Surf',
 #  'Karst - UB_ID',
 'Karst - UB_Name',
 'Karst - UB_Type',
 'Karst - UB_Flux',
 'Karst - UB_Alt_min',
 'Karst - UB_Alt_moy',
 'Karst - UB_Alt_max',
 'Shape_Length_left',
 'Shape_Area_left',
 'Karst - FID_NK',
 #  'Karst - NK_ID',
 'Karst - NK_Hdyn',
 'Karst - NK_Type',
 'Karst - NK_Level',
 'Karst - MNK_ID',
 'Shape_Length_right',
 'Shape_Area_right',
 #  'Forest - Region_de',
 'Forest - Region_fr',
 #  'Forest - Region_it',
 'Forest - Region_en',
 #  'Forest - Code',
 #  'Forest - Subcode',
 #  'Forest - Code_Bu',
 #  'Forest - Code_Fi',
 'Forest - Shape_Leng',
 'Forest - Shape_Area',
 #  'index_right',
 'TBB - CH_CODE_HN',
 #'TBB - CH_BEZ_D',
 'TBB - CH_BEZ_F',
 #'TBB - BFS_NO',
 'TBB - NAME',
 #'TBB - KT_NO',
 #'TBB - KT_KZ',
 'TBB - FLAECHE'
]

In [ ]:
df_regbl = df_regbl[cols]

## Save the final dataset

In [ ]:
df_regbl.to_csv(f"{ROOT}data/final/dataset-v1.0.csv")